In [4]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

In [5]:
# 1. SPATIAL EXPLAINABILITY: GRAD-CAM
# ==========================================
class MultimodalGradCAM:
    """
    Computes Gradient-weighted Class Activation Mapping (Grad-CAM)
    for the 2D Thermal Branch of the Multimodal Battery Health Network.
    """
    def __init__(self, model, last_conv_layer_name="Conv_1"):
        self.model = model
        self.last_conv_layer_name = last_conv_layer_name
        
        # Access the internal MobileNetV2 sub-model
        mobilenet_submodel = self.model.get_layer("mobilenetv2_1.00_224")
        self.conv_layer = mobilenet_submodel.get_layer(self.last_conv_layer_name)

        # Build feature extraction model up to the target conv layer
        self.conv_extractor = tf.keras.Model(
            inputs=mobilenet_submodel.inputs,
            outputs=self.conv_layer.output
        )

    def generate_heatmap(self, img_array, telemetry_array, target_class_idx=None):
        """
        Calculates gradients of target class score w.r.t. conv feature maps.
        """
        # Ensure batch dimensions: (1, 224, 224, 3) and (1, 30, 4)
        if len(img_array.shape) == 3:
            img_tensor = tf.expand_dims(img_array, axis=0)
        else:
            img_tensor = img_array

        if len(telemetry_array.shape) == 2:
            tel_tensor = tf.expand_dims(telemetry_array, axis=0)
        else:
            tel_tensor = telemetry_array

        with tf.GradientTape() as tape:
            # 1. Forward pass through convolutional feature extractor
            conv_outputs = self.conv_extractor(img_tensor)
            tape.watch(conv_outputs)

            # 2. Forward pass through remaining spatial pooling & dense layers
            spatial_pooled = self.model.get_layer("spatial_gap")(conv_outputs)
            spatial_dense = self.model.get_layer("spatial_embedding")(spatial_pooled)
            spatial_out = self.model.get_layer("dropout_2")(spatial_dense)

            # 3. Forward pass through temporal branch
            conv1d_out = self.model.get_layer("conv1d_1")(tel_tensor)
            bilstm_out = self.model.get_layer("bilstm_layer")(conv1d_out)
            temporal_dense = self.model.get_layer("temporal_embedding")(bilstm_out)
            temporal_out = self.model.get_layer("dropout_3")(temporal_dense)

            # 4. Late fusion & output heads
            fused = self.model.get_layer("feature_concatenation")([spatial_out, temporal_out])
            att_weights = self.model.get_layer("fusion_attention_weights")(fused)
            gated = self.model.get_layer("attention_multiplication")([fused, att_weights])
            shared = self.model.get_layer("shared_representation")(gated)
            risk_preds = self.model.get_layer("risk_output")(shared)

            # If no target class specified, pick the highest-scoring risk state
            if target_class_idx is None:
                target_class_idx = tf.argmax(risk_preds[0])
            loss_value = risk_preds[:, target_class_idx]

        # Extract gradients of target score w.r.t. activation maps
        grads = tape.gradient(loss_value, conv_outputs)

        # Global average pooling of gradients gives channel importance weights
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

        # Weighted combination of feature activation maps
        conv_outputs = conv_outputs[0]
        heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)

        # Apply ReLU and normalize between [0, 1]
        heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
        return heatmap.numpy(), int(target_class_idx)

    def overlay_heatmap(self, heatmap, original_img, alpha=0.4, colormap=cv2.COLORMAP_JET):
        """
        Resizes heatmap and blends it over original infrared frame.
        """
        # Ensure image is in uint8 [0, 255]
        if original_img.max() <= 1.0:
            original_img_uint8 = np.uint8(255 * original_img)
        else:
            original_img_uint8 = np.uint8(original_img)

        # Resize heatmap to match image dimensions
        heatmap_resized = cv2.resize(heatmap, (original_img_uint8.shape[1], original_img_uint8.shape[0]))
        heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), colormap)

        # Superimpose heatmap onto original image
        superimposed_img = cv2.addWeighted(original_img_uint8, 1 - alpha, heatmap_colored, alpha, 0)
        return superimposed_img, heatmap_resized

In [6]:
# 2. TEMPORAL EXPLAINABILITY: FEATURE ATTRIBUTION
# ==========================================
class TelemetryFeatureAttributor:
    """
    Computes sensitivity and baseline attribution values for sequential
    telemetry variables (Voltage, Current, Temperature, Resistance)
    relative to Thermal Runaway risk.
    """
    def __init__(self, model, feature_names=None):
        self.model = model
        self.feature_names = feature_names or ["Voltage (V)", "Current (I)", "Temperature (°C)", "Internal Res (mΩ)"]

    def compute_attributions(self, img_tensor, telemetry_tensor, target_class_idx=1, steps=20):
        """
        Approximates feature contributions using integrated gradients
        across the temporal telemetry input.
        """
        # Baseline reference: all zeros (nominal unperturbed telemetry)
        baseline_tel = tf.zeros_like(telemetry_tensor)
        
        # Linear interpolation between baseline and input sample
        alphas = tf.linspace(0.0, 1.0, steps)
        interpolated_inputs = [baseline_tel + alpha * (telemetry_tensor - baseline_tel) for alpha in alphas]
        
        grads_accumulated = tf.zeros_like(telemetry_tensor)

        for interpolated in interpolated_inputs:
            with tf.GradientTape() as tape:
                tape.watch(interpolated)
                preds = self.model({"thermal_image_input": img_tensor, "telemetry_seq_input": interpolated})
                target_score = preds[1][:, target_class_idx]
            
            grads = tape.gradient(target_score, interpolated)
            if grads is not None:
                grads_accumulated += grads

        avg_grads = grads_accumulated / float(steps)
        integrated_gradients = (telemetry_tensor - baseline_tel) * avg_grads

        # Aggregate across sequence length (30 timesteps) to get per-feature impact
        feature_importance = tf.reduce_mean(integrated_gradients, axis=(0, 1)).numpy()
        return dict(zip(self.feature_names, feature_importance))

In [7]:
# 3. VERIFICATION & ARTIFACT GENERATION
# ==========================================
if __name__ == "__main__":
    os.makedirs("artifacts", exist_ok=True)
    model_path = os.path.join("models", "battery_multimodal_model.keras")

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model artifact not found at {model_path}. Run train_multimodal.py first.")

    print(f"Loading trained multimodal model from {model_path}...")
    model = tf.keras.models.load_model(model_path)

    # 1. Create a sample test input
    sample_img = np.random.uniform(0.0, 1.0, size=(224, 224, 3)).astype(np.float32)
    sample_tel = np.random.uniform(0.0, 1.0, size=(30, 4)).astype(np.float32)

    img_tensor = tf.convert_to_tensor(sample_img[np.newaxis, ...])
    tel_tensor = tf.convert_to_tensor(sample_tel[np.newaxis, ...])

    # 2. Run Forward Inference
    soh_pred, risk_pred = model({"thermal_image_input": img_tensor, "telemetry_seq_input": tel_tensor})
    predicted_risk_class = np.argmax(risk_pred[0])
    class_map = {0: "Safe", 1: "Thermal Warning", 2: "Thermal Runaway Critical"}

    print("\n--- Model Inference Verification ---")
    print(f"Predicted Battery SOH: {float(soh_pred[0][0]) * 100:.2f}%")
    print(f"Predicted Hazard State: {class_map[predicted_risk_class]} ({risk_pred[0][predicted_risk_class]*100:.1f}% confidence)")

    # 3. Execute Grad-CAM on Spatial Branch
    print("\nGenerating Spatial Grad-CAM Saliency Map...")
    gradcam = MultimodalGradCAM(model=model, last_conv_layer_name="Conv_1")
    heatmap, active_class = gradcam.generate_heatmap(img_tensor, tel_tensor, target_class_idx=predicted_risk_class)
    superimposed, _ = gradcam.overlay_heatmap(heatmap, sample_img)

    # 4. Execute Temporal Feature Attribution
    print("Computing Electrical Feature Attributions...")
    attributor = TelemetryFeatureAttributor(model=model)
    attributions = attributor.compute_attributions(img_tensor, tel_tensor, target_class_idx=predicted_risk_class)

    # 5. Plot and Save Visual XAI Explanations
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Original simulated IR image
    axes[0].imshow(sample_img)
    axes[0].set_title("Input Thermal IR Frame")
    axes[0].axis("off")

    # Grad-CAM heatmap overlay
    axes[1].imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Grad-CAM: {class_map[active_class]} Hot-Spot")
    axes[1].axis("off")

    # Attribution Bar Chart
    feats = list(attributions.keys())
    scores = list(attributions.values())
    colors = ["#d9534f" if s > 0 else "#5bc0de" for s in scores]
    axes[2].barh(feats, scores, color=colors)
    axes[2].set_title("Sensor Attribution to Risk Prediction")
    axes[2].set_xlabel("Relative Attribution Score")
    axes[2].grid(axis="x", linestyle="--", alpha=0.6)

    plt.tight_layout()
    output_plot = os.path.join("artifacts", "multimodal_xai_diagnostic.png")
    plt.savefig(output_plot, dpi=200)
    plt.close()

    print(f"Visual XAI explanation successfully generated and saved to: {output_plot}")

Loading trained multimodal model from models\battery_multimodal_model.keras...

--- Model Inference Verification ---
Predicted Battery SOH: 74.29%
Predicted Hazard State: Safe (59.0% confidence)

Generating Spatial Grad-CAM Saliency Map...
Computing Electrical Feature Attributions...
Visual XAI explanation successfully generated and saved to: artifacts\multimodal_xai_diagnostic.png
